# NLP Project — Spam / Ham SMS Classification
## Step 1: Converting Text into Machine-Learning-Ready Features

This notebook walks through the **complete preprocessing pipeline** for the SMS Spam Collection dataset.  
By the end you will have numeric feature matrices ready to plug into any ML or neural model.

### What we cover
| Section | What it does |
|---------|-------------|
| 1 | Load the raw dataset |
| 2 | Clean and normalise text |
| 3 | Split into Train / Dev / Test sets |
| 4A | Bag-of-Words (BoW) vectors |
| 4B | TF-IDF vectors |
| 4C | Word2Vec averaged embeddings |
| 4D | BERT / Sentence-Transformer embeddings |
| 5 | Sanity-check with Logistic Regression |
| 6 | Save all feature matrices to disk |

### Why this matters
Different vectorisation methods capture different aspects of language:
- **BoW / TF-IDF** — which words appear (sparse, interpretable)
- **Word2Vec** — what words *mean* in context (dense, semantic)
- **BERT** — deep contextual understanding (dense, most powerful)

## Setup — Install Dependencies

Run this cell once to install all required packages.  
`gensim` is needed for Word2Vec; `sentence-transformers` is needed for BERT.  
Both are optional — comment them out if you only need BoW / TF-IDF.

In [3]:
# Run once — installs all required libraries
!pip install pandas scikit-learn nltk gensim sentence-transformers

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 30.2 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 571.3/571.3 kB 37.7 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 65.1 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 645.5/645.5 kB 33.5 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 57.7 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.0/3.0 MB 40.7 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 80.5/80.5 MB 37.2 MB/s  0:00:02m0:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 39.8 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.3/6.3 MB 60.6 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 536.2/536.2 kB 19.8 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 22/22 [sentence-transformers]ence-transformers]


## Section 0 — Imports

We import all libraries upfront so any missing dependency is caught immediately rather than mid-run.

| Library | Purpose |
|---------|---------|
| `re`, `string` | Regular expressions and punctuation constants for text cleaning |
| `numpy`, `pandas` | Numeric arrays and DataFrame operations |
| `sklearn` | Vectorisers, train/test split, label encoding, evaluation metrics |
| `nltk` | Stopword lists, tokeniser, stemmer, lemmatizer |
| `gensim` | Word2Vec training (imported later, inside the function) |
| `sentence_transformers` | Pre-trained BERT encoding (imported later, inside the function) |

In [4]:
import re
import string
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.preprocessing import LabelEncoder

import nltk
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer, WordNetLemmatizer
from nltk.tokenize import word_tokenize

# Download required NLTK data (only needed once — safe to re-run)
for pkg in ["stopwords", "punkt", "punkt_tab", "wordnet"]:
    nltk.download(pkg, quiet=True)

print("All imports successful.")

All imports successful.


---
## Section 1 — Load Data

### About the dataset
The **SMS Spam Collection** contains **5,574 real SMS messages** in English, each labelled:
- `ham` — legitimate message (86.6 % of the dataset)
- `spam` — unwanted message (13.4 %)

The file is **tab-separated with no header row** — two columns per line:
```
ham\tGo until jurong point, crazy.. Available only...
spam\tFree entry in 2 a wkly comp to win FA Cup final...
```

### What `load_data` does
1. Reads the raw file with `pd.read_csv(sep="\t")`
2. Names the columns `label` and `message`
3. Prints a class-distribution summary so you can spot class imbalance early

> **Class imbalance note:** With ~87% ham and ~13% spam, a naive model that always predicts "ham" would score 87% accuracy — which is *misleading*. This is why we use F1-score, precision, and recall as evaluation metrics, not just accuracy.

In [5]:
def load_data(filepath: str = "SMSSpamCollection.txt") -> pd.DataFrame:
    """
    Load the SMS Spam Collection dataset.
    
    Parameters
    ----------
    filepath : str
        Path to the tab-separated dataset file.
    
    Returns
    -------
    pd.DataFrame with columns: label (str), message (str)
    """
    df = pd.read_csv(
        filepath,
        sep="\t",
        header=None,
        names=["label", "message"],
        encoding="latin-1",   # dataset uses latin-1 encoding
    )
    print(f"Total messages loaded : {len(df):,}")
    print("\nClass distribution:")
    print(df["label"].value_counts().to_string())
    print(f"\nSpam rate: {df['label'].eq('spam').mean():.1%}")
    return df

# ── Run it ────────────────────────────────────────────────────────────────────
df = load_data("SMSSpamCollection.txt")
df.head(5)

Total messages loaded : 5,572

Class distribution:
label
ham     4825
spam     747

Spam rate: 13.4%


,label,message
0,ham,"Go until jurong point, crazy.. Available only ..."
1,ham,Ok lar... Joking wif u oni...
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...
3,ham,U dun say so early hor... U c already then say...
4,ham,"Nah I don't think he goes to usf, he lives aro..."


---
## Section 2 — Text Preprocessing

### Why preprocess at all?
Raw SMS text is noisy: mixed case, URLs, phone numbers, slang, abbreviations.  
Preprocessing **reduces vocabulary size** and helps the model focus on meaningful signal.

### Pipeline overview
Each message passes through these steps in order:

```
Raw text
  ↓  1. Lowercase           "FREE PRIZE" → "free prize"
  ↓  2. Remove URLs          "click http://..." → "click"
  ↓  3. Remove phone nums    "call 07911123" → "call"
  ↓  4. Remove punct/digits  "win!!!" → "win"
  ↓  5. Tokenise             "win prize" → ["win", "prize"]
  ↓  6. Remove stopwords     ["a", "the"] (removes "the", "a", etc.)
  ↓  7. Lemmatize / Stem     ["Plays", "played"] → ["play"]
Clean tokens
```

### Stemming vs Lemmatisation — which to use?
| Technique | What it does | Example | Downside |
|-----------|-------------|---------|---------|
| **Stemming** (Porter) | Chops word endings with rules | "running" → "run" | Can produce non-words: "studies" → "studi" |
| **Lemmatisation** (WordNet) | Maps to dictionary base form | "running" → "run" | Slower, but always a real word |

For SMS spam classification, **lemmatisation** is the safer default.  
Use `use_stemming=True` only if speed is critical.

> **📝 Note on SMS slang:** This dataset contains informal shorthand like `"wif"` (with),
> `"ur"` (your), `"lar"`, `"oni"` — these are **intentionally kept** after preprocessing.
> Although they are not standard English words, they are consistent vocabulary across
> messages and can carry real signal (e.g. `"ur"` appears in both ham and spam, while
> `"wif"` is more ham-specific). Removing them with a dictionary filter would lose that
> signal. If your model struggles, you could experiment with replacing known slang tokens
> with their standard forms using a lookup table.

In [7]:
STOP_WORDS = set(stopwords.words("english"))
stemmer    = PorterStemmer()
lemmatizer = WordNetLemmatizer()


def preprocess_text(
    text: str,
    remove_stopwords: bool = True,
    use_stemming: bool = False,
    use_lemmatization: bool = True,
) -> str:
    """
    Clean and normalise a single SMS message.

    Parameters
    ----------
    text             : Raw SMS string
    remove_stopwords : Remove common English words (the, is, at, ...)
    use_stemming     : Apply Porter stemmer (aggressive, fast)
    use_lemmatization: Apply WordNet lemmatizer (gentler, real words)

    Returns
    -------
    Cleaned string ready for vectorisation.
    """
    # 1. Lowercase — treats "FREE" and "free" as the same word
    text = text.lower()

    # 2. Remove URLs — they add noise, not signal
    text = re.sub(r"http\S+|www\S+", "", text)

    # 3. Remove phone numbers (common spam pattern, but not the word itself)
    text = re.sub(r"\b\d[\d\s\-]{6,}\d\b", "", text)

    # 4. Remove punctuation and digits — keep letters and spaces only
    text = re.sub(r"[^a-z\s]", " ", text)

    # 5. Tokenise into individual words
    tokens = word_tokenize(text)

    # 6. Remove stopwords — high-frequency words that carry little meaning
    if remove_stopwords:
        tokens = [t for t in tokens if t not in STOP_WORDS]

    # 7. Stem or Lemmatize — reduce inflected forms to a base form
    if use_stemming:
        tokens = [stemmer.stem(t) for t in tokens]
    elif use_lemmatization:
        tokens = [lemmatizer.lemmatize(t) for t in tokens]

    return " ".join(tokens)


def preprocess_dataframe(df: pd.DataFrame):
    """
    Apply preprocessing to every row and encode labels.

    Label encoding:  ham → 0,  spam → 1
    """
    df = df.copy()
    df["clean_message"] = df["message"].apply(preprocess_text)

    le = LabelEncoder()
    le.fit(["ham", "spam"])              # explicit ordering: ham=0, spam=1
    df["label_encoded"] = le.transform(df["label"])

    return df, le

# ── Run it ────────────────────────────────────────────────────────────────────
df, label_encoder = preprocess_dataframe(df)

# Show before / after comparison
comparison = df[["label", "message", "clean_message"]].head(6)
for _, row in comparison.iterrows():
    print(f"[{row.label.upper():4s}]  RAW: {row.message[:60]}")
    print(f"        CLEAN: {row.clean_message[:60]}")


[HAM ]  RAW: Go until jurong point, crazy.. Available only in bugis n gre
        CLEAN: go jurong point crazy available bugis n great world la e buf
[HAM ]  RAW: Ok lar... Joking wif u oni...
        CLEAN: ok lar joking wif u oni
[SPAM]  RAW: Free entry in 2 a wkly comp to win FA Cup final tkts 21st Ma
        CLEAN: free entry wkly comp win fa cup final tkts st may text fa re
[HAM ]  RAW: U dun say so early hor... U c already then say...
        CLEAN: u dun say early hor u c already say
[HAM ]  RAW: Nah I don't think he goes to usf, he lives around here thoug
        CLEAN: nah think go usf life around though
[SPAM]  RAW: FreeMsg Hey there darling it's been 3 week's now and no word
        CLEAN: freemsg hey darling week word back like fun still tb ok xxx 


---
## Section 3 — Train / Dev / Test Split

### Why three splits?
| Split | Size | Purpose |
|-------|------|---------|
| **Train** | 60% | Fit the model (and vectoriser) |
| **Dev** | 20% | Tune hyperparameters, compare models |
| **Test** | 20% | Final held-out evaluation — touch this ONCE at the end |

> **Critical rule:** The vectoriser (BoW / TF-IDF) must be **fit only on the training set**, then applied (`.transform()`) to dev and test. Fitting on the full dataset would let training see test-set vocabulary — that's **data leakage** and inflates your scores.

### Stratified splitting
`stratify=y` ensures each split has the same ~13% spam rate as the full dataset.  
Without this, a small split could end up with very few spam examples by chance.

In [8]:
def split_data(df: pd.DataFrame):
    """
    Split into Train (60%) / Dev (20%) / Test (20%) with stratification.
    
    The split is done BEFORE any feature extraction to prevent data leakage.
    
    Returns
    -------
    X_train, X_dev, X_test : numpy arrays of cleaned message strings
    y_train, y_dev, y_test : numpy arrays of integer labels (0=ham, 1=spam)
    """
    X = df["clean_message"].values
    y = df["label_encoded"].values

    # Step 1: carve out 20% for the final test set
    X_traindev, X_test, y_traindev, y_test = train_test_split(
        X, y, test_size=0.20, random_state=42, stratify=y
    )

    # Step 2: split remaining 80% into 75% train / 25% dev → 60/20 overall
    X_train, X_dev, y_train, y_dev = train_test_split(
        X_traindev, y_traindev,
        test_size=0.25, random_state=42, stratify=y_traindev
    )

    n = len(X)
    print("Split sizes (stratified):")
    print(f"  Train : {len(X_train):,}  ({len(X_train)/n:.0%})  spam: {y_train.mean():.1%}")
    print(f"  Dev   : {len(X_dev):,}   ({len(X_dev)/n:.0%})  spam: {y_dev.mean():.1%}")
    print(f"  Test  : {len(X_test):,}   ({len(X_test)/n:.0%})  spam: {y_test.mean():.1%}")

    return X_train, X_dev, X_test, y_train, y_dev, y_test

# ── Run it ────────────────────────────────────────────────────────────────────
X_train, X_dev, X_test, y_train, y_dev, y_test = split_data(df)

Split sizes (stratified):
  Train : 3,342  (60%)  spam: 13.4%
  Dev   : 1,115   (20%)  spam: 13.5%
  Test  : 1,115   (20%)  spam: 13.4%


---
## Section 4A — Bag-of-Words (BoW)

### Concept
BoW treats each message as an **unordered set of word counts**.  
Given a vocabulary of N words, each message becomes a vector of length N  
where position `i` holds the count of word `i`.

```
Vocabulary:  ["call", "free", "win", "hello", "how"]

"free call now"  →  [1, 1, 0, 0, 0]
"win free prize" →  [0, 1, 1, 0, 0]
```

### Key parameters
| Parameter | Value | Why |
|-----------|-------|-----|
| `max_features` | 5,000 | Cap vocabulary to most-frequent 5k terms |
| `ngram_range=(1,2)` | unigrams + bigrams | Captures phrases like "free prize", "call now" |

### Who should use this
The team member implementing **Naïve Bayes from scratch** — the count matrix maps directly onto the likelihood calculations.

In [9]:
def get_bow_features(X_train, X_dev, X_test, max_features: int = 5000):
    """
    Build Bag-of-Words feature matrices using CountVectorizer.

    Each message → sparse vector of word/bigram counts.
    Vectoriser is fit ONLY on training data to prevent leakage.

    Parameters
    ----------
    X_train, X_dev, X_test : arrays of cleaned message strings
    max_features           : vocabulary size cap

    Returns
    -------
    Sparse scipy matrices + fitted vectoriser
    """
    vectorizer = CountVectorizer(
        max_features=max_features,
        ngram_range=(1, 2),    # include both single words and 2-word phrases
        analyzer="word",
        strip_accents="unicode",
    )

    # IMPORTANT: fit_transform on train, transform-only on dev/test
    X_train_bow = vectorizer.fit_transform(X_train)
    X_dev_bow   = vectorizer.transform(X_dev)
    X_test_bow  = vectorizer.transform(X_test)

    feature_names = vectorizer.get_feature_names_out()

    print(f"BoW matrix shape     (train) : {X_train_bow.shape}")
    print(f"Vocabulary size              : {len(feature_names):,}")
    print(f"Matrix density               : {X_train_bow.nnz / (X_train_bow.shape[0]*X_train_bow.shape[1]):.4%}")
    print(f"\nFirst 10 vocabulary terms    : {feature_names[:10].tolist()}")
    print(f"Last  10 vocabulary terms    : {feature_names[-10:].tolist()}")

    return X_train_bow, X_dev_bow, X_test_bow, vectorizer

# ── Run it ────────────────────────────────────────────────────────────────────
X_tr_bow, X_dv_bow, X_te_bow, bow_vec = get_bow_features(X_train, X_dev, X_test)

# Peek at a single message as a vector
sample_idx = 2
print(f"\nSample message   : '{X_train[sample_idx]}'")
row = X_tr_bow[sample_idx]
nonzero_features = [(bow_vec.get_feature_names_out()[i], row[0, i]) for i in row.nonzero()[1]]
print(f"Non-zero entries : {sorted(nonzero_features, key=lambda x: -x[1])[:10]}")

BoW matrix shape     (train) : (3342, 5000)
Vocabulary size              : 5,000
Matrix density               : 0.1873%

First 10 vocabulary terms    : ['aathi', 'abiola', 'able', 'abt', 'ac', 'accept', 'access', 'accident', 'accidentally', 'accordingly']
Last  10 vocabulary terms    : ['yo', 'yor', 'yr', 'yr prize', 'yun', 'yuo', 'yup', 'yup ok', 'zed', 'zed profit']

Sample message   : 'hey babe going ever figure going new year'
Non-zero entries : [('going', np.int64(2)), ('hey', np.int64(1)), ('babe', np.int64(1)), ('ever', np.int64(1)), ('figure', np.int64(1)), ('new', np.int64(1)), ('year', np.int64(1)), ('hey babe', np.int64(1)), ('ever figure', np.int64(1)), ('new year', np.int64(1))]


---
## Section 4B — TF-IDF

### Concept
TF-IDF improves on raw BoW by **downweighting words that appear in many documents**.  
A word like "the" appearing everywhere has low IDF; a word like "winner" appearing  
only in spam messages has high IDF — TF-IDF boosts it accordingly.

**Formula:**
```
TF(t, d)  = count of term t in document d  / total terms in d
IDF(t)    = log( (N + 1) / (df(t) + 1) ) + 1     [smooth variant]
TF-IDF    = TF × IDF
```

### `sublinear_tf=True`
Applies `tf = 1 + log(tf)` instead of raw counts.  
This dampens the effect of a word appearing 50 times vs 5 times — the 10× count difference  
becomes only a 2× score difference. Better for SMS where spam words repeat aggressively.

### When to use TF-IDF vs BoW
| Use BoW when... | Use TF-IDF when... |
|----------------|--------------------|
| Model assumes raw counts (Naïve Bayes) | Model benefits from normalized weights (SVM, Logistic Regression, Neural Net) |
| Corpus is very small | You want common words like "call" downweighted |

### Who should use this
The **SVM** and **Neural Network** team members — TF-IDF typically gives better results with these models.

> **Sanity check caveat:** In the Section 5 sanity check, TF-IDF scores a **lower spam**
> **recall (70%)** than BoW (82%) when paired with Logistic Regression. This is **expected**
> **and not a bug.** Logistic Regression is not the ideal consumer of TF-IDF weights —
> the normalised float values work best with distance-based or margin-based models.
>
> Expected performance with the right model:
>
> | Your model          | Expected spam F1 with TF-IDF |
> |---------------------|------------------------------|
> | Logistic Regression | ~0.82 (sanity check baseline) |
> | SVM (linear kernel) | ~0.95+                        |
> | Neural Net (MLP)    | ~0.93+                        |
>

In [10]:
def get_tfidf_features(X_train, X_dev, X_test, max_features: int = 5000):
    """
    Build TF-IDF feature matrices.

    Downweights very common words and rewards rare informative ones.
    Uses smooth IDF and sublinear TF dampening.

    Returns
    -------
    Sparse scipy matrices + fitted TfidfVectorizer
    """
    vectorizer = TfidfVectorizer(
        max_features=max_features,
        ngram_range=(1, 2),      # unigrams + bigrams
        sublinear_tf=True,       # log(1 + tf) instead of raw tf
        analyzer="word",
        strip_accents="unicode",
    )

    X_train_tfidf = vectorizer.fit_transform(X_train)
    X_dev_tfidf   = vectorizer.transform(X_dev)
    X_test_tfidf  = vectorizer.transform(X_test)

    feature_names = vectorizer.get_feature_names_out()
    idf_scores    = vectorizer.idf_

    print(f"TF-IDF matrix shape  (train) : {X_train_tfidf.shape}")
    print(f"Matrix density               : {X_train_tfidf.nnz / (X_train_tfidf.shape[0]*X_train_tfidf.shape[1]):.4%}")

    # Show the 10 highest-IDF words (rarest, most distinctive)
    top_idf_idx = np.argsort(idf_scores)[-10:][::-1]
    print("\nTop 10 highest-IDF terms (most distinctive):")
    for idx in top_idf_idx:
        print(f"  {feature_names[idx]:25s}  idf={idf_scores[idx]:.3f}")

    # Show the 10 lowest-IDF words (most common, downweighted)
    bot_idf_idx = np.argsort(idf_scores)[:10]
    print("\nBottom 10 lowest-IDF terms (most common, downweighted):")
    for idx in bot_idf_idx:
        print(f"  {feature_names[idx]:25s}  idf={idf_scores[idx]:.3f}")

    return X_train_tfidf, X_dev_tfidf, X_test_tfidf, vectorizer

# ── Run it ────────────────────────────────────────────────────────────────────
X_tr_tfidf, X_dv_tfidf, X_te_tfidf, tfidf_vec = get_tfidf_features(X_train, X_dev, X_test)

TF-IDF matrix shape  (train) : (3342, 5000)
Matrix density               : 0.1873%

Top 10 highest-IDF terms (most distinctive):
  donate                     idf=8.421
  horo                       idf=8.421
  joy father                 idf=8.421
  pan                        idf=8.421
  stick                      idf=8.421
  jsco                       idf=8.421
  aom                        idf=8.421
  make forget                idf=8.421
  make girl                  idf=8.421
  nahi                       idf=8.421

Bottom 10 lowest-IDF terms (most common, downweighted):
  call                       idf=3.234
  get                        idf=3.753
  ur                         idf=3.868
  go                         idf=3.950
  day                        idf=3.961
  ok                         idf=4.058
  got                        idf=4.091
  know                       idf=4.124
  good                       idf=4.152
  gt                         idf=4.152


---
## Section 4C — Word2Vec Averaged Embeddings

### Concept
Word2Vec learns a **dense vector** (embedding) for each word by training a shallow  
neural network to predict context words. Words with similar meanings end up  
close together in vector space:

```
king − man + woman ≈ queen
free − legitimate + helpful ≈ useful
```

### Averaging strategy
Since each SMS has a different number of words, we produce a **fixed-length  document vector** by averaging all word vectors in the message:

```
"win free prize"  →  (vec("win") + vec("free") + vec("prize")) / 3
```

This loses word order but captures **semantic meaning** — something BoW/TF-IDF cannot do.

### Key parameters
| Parameter | Value | What it controls |
|-----------|-------|-----------------|
| `vector_size` | 100 | Dimension of each word vector |
| `window` | 5 | How many surrounding words define "context" |
| `min_count` | 2 | Ignore words appearing fewer than 2 times |
| `sg=1` | Skip-gram | Predicts context from centre word (better for rare words) |

### Who should use this
The team member implementing a **neural network with dense inputs** — Word2Vec vectors  
are 100-dimensional dense arrays, perfect for feed-forward or LSTM networks.

In [11]:
def get_word2vec_features(X_train, X_dev, X_test, embedding_dim: int = 100):
    """
    Train Word2Vec on the training corpus, then average word vectors per message.

    Produces dense numpy arrays of shape (n_samples, embedding_dim).
    
    Word2Vec is trained ONLY on X_train — dev/test words not in the training
    vocabulary get zero vectors (out-of-vocabulary handling).

    Parameters
    ----------
    embedding_dim : int
        Size of each word vector (100 is a good default for short texts).

    Returns
    -------
    Dense numpy arrays (n_samples, 100) + trained Word2Vec model
    """
    try:
        from gensim.models import Word2Vec
    except ImportError:
        print("gensim not installed. Run: pip install gensim")
        return None, None, None, None

    # Word2Vec expects a list of token lists, not strings
    tokenized_train = [text.split() for text in X_train]

    print("Training Word2Vec on training corpus...")
    w2v_model = Word2Vec(
        sentences=tokenized_train,
        vector_size=embedding_dim,
        window=5,               # context window size
        min_count=2,            # ignore very rare words
        workers=4,              # parallel training
        sg=1,                   # 1=skip-gram, 0=CBOW
        epochs=15,
        seed=42,
    )
    print(f"Vocabulary size (Word2Vec) : {len(w2v_model.wv):,} words")

    def average_embedding(tokens):
        """Average vectors for all known words; return zeros for OOV messages."""
        vecs = [w2v_model.wv[t] for t in tokens if t in w2v_model.wv]
        if vecs:
            return np.mean(vecs, axis=0)
        return np.zeros(embedding_dim)   # out-of-vocabulary fallback

    def embed_corpus(texts):
        return np.array([average_embedding(t.split()) for t in texts])

    X_train_w2v = embed_corpus(X_train)
    X_dev_w2v   = embed_corpus(X_dev)
    X_test_w2v  = embed_corpus(X_test)

    print(f"Word2Vec feature shape (train) : {X_train_w2v.shape}")
    print(f"                      (dev)   : {X_dev_w2v.shape}")

    # Show most similar words to 'free' and 'win' — classic spam triggers
    for anchor in ["free", "win", "call"]:
        if anchor in w2v_model.wv:
            similar = w2v_model.wv.most_similar(anchor, topn=5)
            print(f"\nWords most similar to '{anchor}':")
            for word, score in similar:
                print(f"  {word:20s} {score:.3f}")

    return X_train_w2v, X_dev_w2v, X_test_w2v, w2v_model

# ── Run it ────────────────────────────────────────────────────────────────────
X_tr_w2v, X_dv_w2v, X_te_w2v, w2v_model = get_word2vec_features(X_train, X_dev, X_test)

/opt/anaconda3/envs/ml/lib/python3.11/site-packages/requests/__init__.py:86: RequestsDependencyWarning: Unable to find acceptable character detection dependency (chardet or charset_normalizer).
  warnings.warn(


Training Word2Vec on training corpus...


Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'


Vocabulary size (Word2Vec) : 2,611 words
Word2Vec feature shape (train) : (3342, 100)
                      (dev)   : (1115, 100)

Words most similar to 'free':
  update               0.937
  deliveredtomorrow    0.933
  colour               0.932
  latest               0.930
  mths                 0.925

Words most similar to 'win':
  wkly                 0.923
  chance               0.913
  comp                 0.903
  ipod                 0.897
  entry                0.888

Words most similar to 'call':
  match                0.849
  pm                   0.847
  landline             0.843
  quoting              0.842
  line                 0.841


---
## Section 4D — BERT / Sentence-Transformer Embeddings

### Concept
BERT (Bidirectional Encoder Representations from Transformers) is a **pre-trained  
language model** that produces contextual embeddings — the same word gets a  
different vector depending on its surrounding words:

```
"I need to win this"   →  "win" vector reflects uncertain desire
"Win £500 NOW call"    →  "win" vector reflects spam urgency
```

We use the lightweight **`all-MiniLM-L6-v2`** model from `sentence-transformers`,  
which produces **384-dimensional** dense vectors with high quality in ~10s on CPU.

### Word2Vec vs BERT
| | Word2Vec | BERT |
|--|----------|------|
| Context | None — each word has one fixed vector | Full sentence context |
| Training | On this SMS corpus | Pre-trained on 1B+ sentences |
| Vector size | 100 | 384 |
| Speed | Fast | ~10s for 5,574 messages |
| Best for | Models where semantic clusters matter | State-of-the-art results |

### Who should use this
Anyone wanting the **strongest possible baseline** — BERT features typically  
outperform BoW and TF-IDF with the same classifier on top.

In [12]:
def get_bert_features(X_train, X_dev, X_test,
                      model_name: str = "all-MiniLM-L6-v2"):
    """
    Encode messages with a pre-trained Sentence-BERT model.

    Each message → dense vector of 384 dimensions.
    No training required — the model is already pre-trained.
    
    Parameters
    ----------
    model_name : str
        Hugging Face model identifier. 'all-MiniLM-L6-v2' is fast and accurate.

    Returns
    -------
    Dense numpy arrays (n_samples, 384) + loaded model
    """
    try:
        from sentence_transformers import SentenceTransformer
    except ImportError:
        print("sentence-transformers not installed.  Run: pip install sentence-transformers")
        return None, None, None, None

    print(f"Loading BERT model '{model_name}'...")
    bert = SentenceTransformer(model_name)

    print("Encoding training messages...")
    X_train_bert = bert.encode(
        list(X_train), show_progress_bar=True, batch_size=64, convert_to_numpy=True
    )
    print("Encoding dev messages...")
    X_dev_bert   = bert.encode(
        list(X_dev),   show_progress_bar=True, batch_size=64, convert_to_numpy=True
    )
    print("Encoding test messages...")
    X_test_bert  = bert.encode(
        list(X_test),  show_progress_bar=True, batch_size=64, convert_to_numpy=True
    )

    print(f"\nBERT feature shape (train) : {X_train_bert.shape}")
    print(f"               (dev)     : {X_dev_bert.shape}")

    # Show cosine similarity between a ham and a spam example
    from numpy.linalg import norm
    def cosine_sim(a, b): return np.dot(a,b) / (norm(a) * norm(b))
    spam_idx = np.where(y_train == 1)[0][0]
    ham_idx  = np.where(y_train == 0)[0][0]
    sim = cosine_sim(X_train_bert[spam_idx], X_train_bert[ham_idx])
    print(f"\nCosine similarity (spam vs ham example) : {sim:.4f}")
    print("  (lower = more distinct — BERT distinguishes spam/ham semantically)")

    return X_train_bert, X_dev_bert, X_test_bert, bert

# ── Run it (requires sentence-transformers) ───────────────────────────────────
X_tr_bert, X_dv_bert, X_te_bert, bert_model = get_bert_features(X_train, X_dev, X_test)

Loading BERT model 'all-MiniLM-L6-v2'...


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

Encoding dev messages...


Batches:   0%|          | 0/18 [00:00<?, ?it/s]

Encoding test messages...


Batches:   0%|          | 0/18 [00:00<?, ?it/s]


BERT feature shape (train) : (3342, 384)
               (dev)     : (1115, 384)

Cosine similarity (spam vs ham example) : 0.0796
  (lower = more distinct — BERT distinguishes spam/ham semantically)


---
## Section 5 — Sanity Check with Logistic Regression

### Purpose
Before spending time training your actual model, use a fast **Logistic Regression**  
as a sanity check. If your features are well-constructed you should see:
- **BoW / TF-IDF**: ≥ 97% accuracy, ≥ 0.90 F1 on spam
- **Word2Vec**: ≥ 95% accuracy
- **BERT**: ≥ 98% accuracy

If scores are much lower, something went wrong upstream (leakage, wrong split, etc.).

### Reading the classification report
```
              precision  recall  f1-score

ham               0.99    0.99      0.99    ← most messages, easy to get right
spam              0.95    0.93      0.94    ← the class we care about

accuracy                            0.98
```

- **Precision** for spam: of all messages predicted spam, how many really are?
- **Recall** for spam: of all actual spam, how many did we catch?
- **F1** is the harmonic mean — use this as your primary metric for comparison.

In [13]:
def evaluate_features(X_tr, X_dv, y_tr, y_dv, name: str):
    """
    Fit a Logistic Regression and print a classification report on the dev set.
    
    Used purely as a sanity check — not the final model.
    
    Parameters
    ----------
    X_tr, X_dv : feature matrices (sparse or dense)
    y_tr, y_dv : integer label arrays
    name       : label for printing

    Returns
    -------
    Fitted LogisticRegression classifier
    """
    from sklearn.linear_model import LogisticRegression
    from sklearn.metrics import classification_report, confusion_matrix

    clf = LogisticRegression(max_iter=1000, random_state=42, C=1.0)
    clf.fit(X_tr, y_tr)
    y_pred = clf.predict(X_dv)

    print(f"\n{'='*50}")
    print(f"  {name}")
    print(f"{'='*50}")
    print(classification_report(y_dv, y_pred, target_names=["ham", "spam"]))

    cm = confusion_matrix(y_dv, y_pred)
    print(f"Confusion matrix:")
    print(f"  TN={cm[0,0]}  FP={cm[0,1]}  (ham predicted as spam)")
    print(f"  FN={cm[1,0]}  TP={cm[1,1]}  (spam caught correctly)")

    return clf

# ── Run sanity checks ─────────────────────────────────────────────────────────
print("Running sanity checks on Dev set...")
_ = evaluate_features(X_tr_bow,   X_dv_bow,   y_train, y_dev, "Bag-of-Words (BoW)")
_ = evaluate_features(X_tr_tfidf, X_dv_tfidf, y_train, y_dev, "TF-IDF")

if X_tr_w2v is not None:
    _ = evaluate_features(X_tr_w2v, X_dv_w2v, y_train, y_dev, "Word2Vec avg embedding")

if X_tr_bert is not None:
    _ = evaluate_features(X_tr_bert, X_dv_bert, y_train, y_dev, "BERT (Sentence-Transformer)")

Running sanity checks on Dev set...

  Bag-of-Words (BoW)
              precision    recall  f1-score   support

         ham       0.97      1.00      0.99       965
        spam       0.99      0.82      0.90       150

    accuracy                           0.97      1115
   macro avg       0.98      0.91      0.94      1115
weighted avg       0.98      0.97      0.97      1115

Confusion matrix:
  TN=964  FP=1  (ham predicted as spam)
  FN=27  TP=123  (spam caught correctly)

  TF-IDF
              precision    recall  f1-score   support

         ham       0.96      1.00      0.98       965
        spam       0.98      0.70      0.82       150

    accuracy                           0.96      1115
   macro avg       0.97      0.85      0.90      1115
weighted avg       0.96      0.96      0.95      1115

Confusion matrix:
  TN=963  FP=2  (ham predicted as spam)
  FN=45  TP=105  (spam caught correctly)

  Word2Vec avg embedding
              precision    recall  f1-score   support


---
## Section 6 — Save Feature Matrices for Team Use

### Why save to disk?
Each team member builds a different model but needs the **same feature matrices**  
so results are comparable on the same held-out test set.

Saving once here means:
1. No one re-runs preprocessing differently
2. No accidental data leakage from re-fitting the vectoriser
3. Faster iteration — loading `.npz` is instant vs re-vectorising

### File format
- **Sparse matrices** (BoW, TF-IDF) → `scipy.sparse.save_npz` → `.npz`  
- **Dense arrays** (Word2Vec, BERT, labels) → `numpy.save` → `.npy`

### How to load in your model notebook
```python
import scipy.sparse as sp
import numpy as np

X_train = sp.load_npz("X_train_tfidf.npz")   # sparse matrix
y_train = np.load("y_train.npy")              # labels

# For dense representations:
X_train_w2v  = np.load("X_train_w2v.npy")
X_train_bert = np.load("X_train_bert.npy")
```

In [14]:
import scipy.sparse as sp

print("Saving feature matrices...")

# ── Sparse matrices (BoW and TF-IDF) ─────────────────────────────────────────
sp.save_npz("X_train_bow.npz",   X_tr_bow)
sp.save_npz("X_dev_bow.npz",     X_dv_bow)
sp.save_npz("X_test_bow.npz",    X_te_bow)
print("Saved BoW matrices.")

sp.save_npz("X_train_tfidf.npz", X_tr_tfidf)
sp.save_npz("X_dev_tfidf.npz",   X_dv_tfidf)
sp.save_npz("X_test_tfidf.npz",  X_te_tfidf)
print("Saved TF-IDF matrices.")

# ── Dense arrays (Word2Vec, BERT) ─────────────────────────────────────────────
if X_tr_w2v is not None:
    np.save("X_train_w2v.npy", X_tr_w2v)
    np.save("X_dev_w2v.npy",   X_dv_w2v)
    np.save("X_test_w2v.npy",  X_te_w2v)
    print("Saved Word2Vec arrays.")

if X_tr_bert is not None:
    np.save("X_train_bert.npy", X_tr_bert)
    np.save("X_dev_bert.npy",   X_dv_bert)
    np.save("X_test_bert.npy",  X_te_bert)
    print("Saved BERT arrays.")

# ── Labels ────────────────────────────────────────────────────────────────────
np.save("y_train.npy", y_train)
np.save("y_dev.npy",   y_dev)
np.save("y_test.npy",  y_test)
print("Saved label arrays.")

print("\n" + "="*50)
print("  All files saved. Summary:")
print("="*50)
import os
for fname in sorted(os.listdir(".")):
    if fname.endswith((".npz", ".npy")):
        size_kb = os.path.getsize(fname) / 1024
        print(f"  {fname:35s}  {size_kb:7.1f} KB")

print("\nQuick reload verification:")
X_check = sp.load_npz("X_train_tfidf.npz")
y_check = np.load("y_train.npy")
print(f"  X_train_tfidf shape : {X_check.shape}  ✓")
print(f"  y_train shape       : {y_check.shape}  ✓")

Saving feature matrices...
Saved BoW matrices.
Saved TF-IDF matrices.
Saved Word2Vec arrays.
Saved BERT arrays.
Saved label arrays.

  All files saved. Summary:
  X_dev_bert.npy                        1672.6 KB
  X_dev_bow.npz                           19.2 KB
  X_dev_tfidf.npz                         75.6 KB
  X_dev_w2v.npy                          871.2 KB
  X_test_bert.npy                       1672.6 KB
  X_test_bow.npz                          19.0 KB
  X_test_tfidf.npz                        75.2 KB
  X_test_w2v.npy                         871.2 KB
  X_train_bert.npy                      5013.1 KB
  X_train_bow.npz                         63.3 KB
  X_train_tfidf.npz                      244.0 KB
  X_train_w2v.npy                       2611.1 KB
  y_dev.npy                                8.8 KB
  y_test.npy                               8.8 KB
  y_train.npy                             26.2 KB

Quick reload verification:
  X_train_tfidf shape : (3342, 5000)  ✓
  y_train shape      

---
## Summary — Which Features to Use for Each Model

| Team member | Input features | File to load |
|-------------|---------------|--------------|
| **Naïve Bayes (scratch)** | BoW counts | `X_train_bow.npz` |
| **SVM** | TF-IDF + handcrafted | `X_train_tfidf.npz` + build extra features |
| **Neural net (sparse)** | TF-IDF | `X_train_tfidf.npz` |
| **Neural net (dense)** | Word2Vec avg | `X_train_w2v.npy` |
| **Best possible baseline** | BERT | `X_train_bert.npy` |
